In [15]:
import pandas as pd
economics = pd.read_csv('economics.csv', sep=';')
print(economics.head())

  codigo_residencia    fecha  ingresos  costes
0              R001  01/2023     45795   15860
1              R001  02/2023     68158   59732
2              R001  03/2023     41284   21265
3              R001  04/2023     46850   19426
4              R001  05/2023     51962   29423


In [16]:
import pandas as pd
maestro_residencias = pd.read_csv('maestro_residencias.csv', sep=',')
print(maestro_residencias.head())

  codigo_residencia   nombre_residencia
0              R001                 ---
1              R002           As Burgas
2              R003  Barcelona Diagonal
3              R004       Blas de Otero
4              R005        Campo Grande


In [17]:
import pandas as pd
sizing = pd.read_csv('sizing.csv', sep=';')
print(sizing.head())

  codigo_residencia  num_rooms room_type
0              R001        112         S
1              R001        102         D
2              R001         24         Q
3              R002        116         S
4              R002         81         D


In [24]:
import pandas as pd
leads_contacts = pd.read_csv('Leads_Contacts.csv', sep=';')
print(leads_contacts.head())

C:\Users\rmendoza\AppData\Local\Temp\ipykernel_24620\1015454137.py:2: DtypeWarning: Columns (0: Particular o Grupo, 1: Residencia escogida) have mixed types. Specify dtype option on import or set low_memory=False.
  leads_contacts = pd.read_csv('Leads_Contacts.csv', sep=';')


  Residencias de interés Residencia actual  \
0             Cualquiera               ---   
1             Cualquiera               ---   
2             Cualquiera               ---   
3             Cualquiera               ---   
4             Cualquiera               ---   

                     Correo electrónico Origen/Campaña Posibles Clientes  \
0           mrojassalvatierra@gmail.com                    SEO & Directo   
1  burcin.atilganturkmen@bilecik.edu.tr                    SEO & Directo   
2                 milobejon@hotmail.com                    SEO & Directo   
3                     pietatp@gmail.com                    SEO & Directo   
4                 dolorspelay@gmail.com                    SEO & Directo   

  Fuente de Posible Cliente Curso Ciudades de interés Particular o Grupo  \
0          Chat no iniciado   NaN          Cualquiera                NaN   
1          Chat no iniciado   NaN          Cualquiera                NaN   
2          Chat no iniciado   NaN     

In [18]:
import pandas as pd

if "economics" not in globals():
    resultado = "La tabla economics no está disponible en el entorno."
elif "maestro_residencias" not in globals():
    resultado = "La tabla maestro_residencias no está disponible en el entorno."
else:
    eco = economics.copy()
    eco["_fecha"] = pd.to_datetime(eco["fecha"], format="%m/%Y", errors="coerce")
    eco_2024 = eco[eco["_fecha"].dt.year == 2024]

    eco_residencia = eco_2024.merge(
        maestro_residencias[["codigo_residencia", "nombre_residencia"]],
        on="codigo_residencia",
        how="inner"
    )

    filtrado = eco_residencia[
        eco_residencia["nombre_residencia"].fillna("").astype(str).str.casefold() == "as burgas".casefold()
    ]

    resultado = filtrado["ingresos"].sum()

In [19]:
resultado

np.int64(574019)

In [20]:
filtrado

,codigo_residencia,fecha,ingresos,costes,_fecha,nombre_residencia
12,R002,01/2024,35056,29948,2024-01-01,As Burgas
13,R002,02/2024,38110,28773,2024-02-01,As Burgas
14,R002,03/2024,57266,32412,2024-03-01,As Burgas
15,R002,04/2024,63270,21910,2024-04-01,As Burgas
16,R002,05/2024,55446,15206,2024-05-01,As Burgas
17,R002,06/2024,51518,37361,2024-06-01,As Burgas
18,R002,07/2024,53419,37403,2024-07-01,As Burgas
19,R002,08/2024,48141,29820,2024-08-01,As Burgas
20,R002,09/2024,36374,21892,2024-09-01,As Burgas
21,R002,10/2024,31678,18242,2024-10-01,As Burgas


In [21]:
import pandas as pd

if "economics" not in globals() or "sizing" not in globals():
    resultado = "La tabla economics o sizing no está disponible en el entorno."
else:
    eco = economics.copy()
    eco["fecha_dt"] = pd.to_datetime(eco["fecha"], format="%m/%Y", errors="coerce")

    ingresos_2024 = (
        eco[eco["fecha_dt"].dt.year == 2024]
        .groupby("codigo_residencia", as_index=False)["ingresos"]
        .sum()
        .rename(columns={"ingresos": "Ingresos anuales 2024"})
    )

    habitaciones = (
        sizing
        .groupby("codigo_residencia", as_index=False)["num_rooms"]
        .sum()
        .rename(columns={"num_rooms": "Habitaciones totales"})
    )

    datos = ingresos_2024.merge(
        habitaciones,
        on="codigo_residencia",
        how="inner"
    ).dropna(subset=["Habitaciones totales", "Ingresos anuales 2024"])

    if len(datos) < 2:
        resultado = "No hay suficientes residencias para calcular la correlación."
    else:
        resultado = datos["Habitaciones totales"].corr(
            datos["Ingresos anuales 2024"],
            method="pearson"
        )

In [22]:
resultado

np.float64(0.1635895503311359)

In [25]:
import pandas as pd

if "leads_contacts" not in globals() or "economics" not in globals():
    resultado = "La tabla leads_contacts o economics no está disponible en el entorno."
else:
    lc = leads_contacts.copy()

    lc_curso = lc[
        lc["Curso_corregido"] == "2024/2025"
    ].copy()

    convertidos = (
        lc_curso[
            (lc_curso["Tipo de registro"] == "Contacts") &
            (lc_curso["Particular o Grupo"] == "Particular")
        ]
        .groupby("Residencias_interes_corregido", dropna=False)["Correo electrónico"]
        .nunique()
        .reset_index(name="Convertidos")
    )

    contacts_particulares = (
        lc_curso[
            (lc_curso["Tipo de registro"] == "Contacts") &
            (lc_curso["Particular o Grupo"] == "Particular")
        ]
        .groupby("Residencias_interes_corregido", dropna=False)["Correo electrónico"]
        .nunique()
        .reset_index(name="Contacts particulares")
    )

    leads = (
        lc_curso[
            lc_curso["Tipo de registro"] == "Leads"
        ]
        .groupby("Residencias_interes_corregido", dropna=False)["Correo electrónico"]
        .nunique()
        .reset_index(name="Leads")
    )

    cr_residencias = contacts_particulares.merge(
        leads,
        on="Residencias_interes_corregido",
        how="outer"
    ).merge(
        convertidos,
        on="Residencias_interes_corregido",
        how="outer"
    )

    cr_residencias[["Contacts particulares", "Leads", "Convertidos"]] = (
        cr_residencias[["Contacts particulares", "Leads", "Convertidos"]].fillna(0)
    )

    denominador = (
        cr_residencias["Contacts particulares"] +
        cr_residencias["Leads"]
    )

    cr_residencias["CR ConversionRate"] = (
        cr_residencias["Convertidos"]
        .div(denominador.where(denominador > 0, 1))
        .where(denominador > 0, 0.0)
    )

    eco = economics.copy()
    eco["fecha_dt"] = pd.to_datetime(
        eco["fecha"],
        format="%m/%Y",
        errors="coerce"
    )

    inicio = pd.Timestamp("2024-09-01")
    fin = pd.Timestamp("2025-08-01")

    ingresos_periodo = (
        eco[
            (eco["fecha_dt"] >= inicio) &
            (eco["fecha_dt"] <= fin)
        ]
        .groupby("codigo_residencia", as_index=False)["ingresos"]
        .sum()
        .rename(columns={"ingresos": "Ingresos septiembre 2024-agosto 2025"})
    )

    datos = cr_residencias.merge(
        ingresos_periodo,
        left_on="Residencias_interes_corregido",
        right_on="codigo_residencia",
        how="inner"
    ).dropna(
        subset=["CR ConversionRate", "Ingresos septiembre 2024-agosto 2025"]
    )

    if len(datos) < 2:
        resultado = "No hay suficientes residencias para calcular la correlación."
    else:
        resultado = datos["CR ConversionRate"].corr(
            datos["Ingresos septiembre 2024-agosto 2025"],
            method="pearson"
        )

In [26]:
datos

,Residencias_interes_corregido,Contacts particulares,Leads,Convertidos,CR ConversionRate,codigo_residencia,Ingresos septiembre 2024-agosto 2025
0,R002,1.0,90,1.0,0.010989,R002,538059
1,R003,7.0,332,7.0,0.020649,R003,627956
2,R004,6.0,406,6.0,0.014563,R004,596569
3,R005,0.0,2,0.0,0.000000,R005,566835
4,R006,2.0,321,2.0,0.006192,R006,656052
5,R007,5.0,158,5.0,0.030675,R007,594731
6,R008,4.0,346,4.0,0.011429,R008,572506
7,R010,1.0,1,1.0,0.500000,R010,589019
8,R011,13.0,215,13.0,0.057018,R011,542355
9,R012,5.0,450,5.0,0.010989,R012,537765
